<a href="https://colab.research.google.com/github/xiyuan1avery/ma2288/blob/research-v2/notebooks/19_v2_direct_acceptance_objectives.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Comparing Decoder KL, Local Acceptance, and Sequence Survival Objectives

Notebook 18 showed that reducing decoder output KL did not reliably
increase fresh accepted-prefix length.

This notebook compares three objectives under the same initialization,
training data, architecture, and optimization budget:

1. Decoder KL
2. Local LK / direct acceptance
3. Sequence survival-LK

For target distribution $p_k$ and draft distribution $q_k$, the
expected speculative acceptance probability is

$$
A_k
=
\sum_x \min\left(p_k(x), q_k(x)\right)
=
1-\mathrm{TV}\left(p_k,q_k\right).
$$

The local objective optimizes each $A_k$ independently.

The sequence objective optimizes

$$
\sum_{k=2}^{K}
\prod_{i=2}^{k} A_i,
$$

which corresponds to the expected number of accepted draft tokens
beyond the guaranteed first token.

## Primary hypothesis

> Under limited draft-model capacity, sequence-level survival
> optimization allocates capacity toward early, reachable positions and
> improves accepted-prefix length more than local distribution matching.

In [1]:
!pip -q install transformers datasets pandas matplotlib seaborn scipy

In [2]:
import os
import json
import copy
import gc
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("PyTorch version:", torch.__version__)
print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch version: 2.11.0+cu128
Device: cuda
GPU: Tesla T4


In [3]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [4]:
PROJECT_DIR = Path(
    "/content/drive/MyDrive/ma2288_nextlat"
)

BASE_CHECKPOINT_PATH = (
    PROJECT_DIR
    / "checkpoints"
    / "multistep_transition_seed42.pt"
)

TEST_PROMPT_PATH = (
    PROJECT_DIR
    / "data_v2"
    / "fresh_wikitext103_prompts_seed42.pt"
)

TRAINING_DATA_DIR = (
    PROJECT_DIR
    / "data_v3"
)

CHECKPOINT_OUTPUT_DIR = (
    PROJECT_DIR
    / "checkpoints_v3"
)

RESULT_TABLE_DIR = (
    PROJECT_DIR
    / "results_v3"
    / "tables"
)

RESULT_FIGURE_DIR = (
    PROJECT_DIR
    / "results_v3"
    / "figures"
)

for directory in [
    TRAINING_DATA_DIR,
    CHECKPOINT_OUTPUT_DIR,
    RESULT_TABLE_DIR,
    RESULT_FIGURE_DIR,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )

assert PROJECT_DIR.exists()
assert BASE_CHECKPOINT_PATH.exists()
assert TEST_PROMPT_PATH.exists()

print("Base checkpoint:")
print(BASE_CHECKPOINT_PATH)

print("\nIndependent test prompts:")
print(TEST_PROMPT_PATH)

print("\nOutput directory:")
print(CHECKPOINT_OUTPUT_DIR)

Base checkpoint:
/content/drive/MyDrive/ma2288_nextlat/checkpoints/multistep_transition_seed42.pt

Independent test prompts:
/content/drive/MyDrive/ma2288_nextlat/data_v2/fresh_wikitext103_prompts_seed42.pt

Output directory:
/content/drive/MyDrive/ma2288_nextlat/checkpoints_v3


In [5]:
MODEL_NAME = "distilgpt2"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

target_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME
).to(device)

target_model.eval()

for parameter in target_model.parameters():
    parameter.requires_grad = False

token_embedding = target_model.get_input_embeddings()

HIDDEN_SIZE = target_model.config.n_embd
VOCAB_SIZE = target_model.config.vocab_size

print("Model:", MODEL_NAME)
print("Hidden size:", HIDDEN_SIZE)
print("Vocabulary size:", VOCAB_SIZE)
print("EOS token ID:", tokenizer.eos_token_id)

print(
    "Trainable target parameters:",
    sum(
        parameter.numel()
        for parameter in target_model.parameters()
        if parameter.requires_grad
    ),
)

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  353MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Model: distilgpt2
Hidden size: 768
Vocabulary size: 50257
EOS token ID: 50256
Trainable target parameters: 0


In [6]:
class ResidualLatentTransition(nn.Module):
    def __init__(
        self,
        hidden_size=768,
        token_embedding_size=768,
        intermediate_size=512,
    ):
        super().__init__()

        input_size = (
            hidden_size + token_embedding_size
        )

        self.input_normalization = nn.LayerNorm(
            input_size
        )

        self.network = nn.Sequential(
            nn.Linear(
                input_size,
                intermediate_size,
            ),
            nn.GELU(),
            nn.Linear(
                intermediate_size,
                hidden_size,
            ),
        )

    def forward(
        self,
        hidden_state,
        token_embedding_value,
    ):
        model_input = torch.cat(
            [
                hidden_state,
                token_embedding_value,
            ],
            dim=-1,
        )

        normalized_input = self.input_normalization(
            model_input
        )

        delta = self.network(
            normalized_input
        )

        return hidden_state + delta

In [7]:
base_checkpoint = torch.load(
    BASE_CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False,
)

print("Checkpoint keys:")
print(base_checkpoint.keys())

assert "model_state_dict" in base_checkpoint

base_state_dict = base_checkpoint[
    "model_state_dict"
]

reference_model = ResidualLatentTransition(
    hidden_size=HIDDEN_SIZE,
    token_embedding_size=HIDDEN_SIZE,
    intermediate_size=512,
).to(device)

load_result = reference_model.load_state_dict(
    base_state_dict,
    strict=True,
)

assert len(load_result.missing_keys) == 0
assert len(load_result.unexpected_keys) == 0

print("\nCheckpoint load result:")
print(load_result)

print(
    "Trainable transition parameters:",
    sum(
        parameter.numel()
        for parameter in reference_model.parameters()
        if parameter.requires_grad
    ),
)

Checkpoint keys:
dict_keys(['model_state_dict', 'model_name', 'hidden_dimension', 'bottleneck_dimension', 'seed', 'rollout_length', 'learning_rate', 'initial_checkpoint', 'best_validation_rollout_loss', 'training_history'])

Checkpoint load result:
<All keys matched successfully>
Trainable transition parameters: 1184000


In [8]:
wikitext_train = load_dataset(
    "Salesforce/wikitext",
    "wikitext-103-raw-v1",
    split="train",
)

wikitext_validation = load_dataset(
    "Salesforce/wikitext",
    "wikitext-103-raw-v1",
    split="validation",
)

print("Train rows:", len(wikitext_train))
print("Validation rows:", len(wikitext_validation))

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

wikitext-103-raw-v1/test-00000-of-00001.(…): reconstructing file:   0%|          |  0.00B /  733kB            

wikitext-103-raw-v1/test-00000-of-00001.(…): downloading bytes:           |  0.00B            

wikitext-103-raw-v1/train-00000-of-00002(…): reconstructing file:   0%|          |  0.00B /  157MB            

wikitext-103-raw-v1/train-00000-of-00002(…): downloading bytes:           |  0.00B            

...

wikitext-103-raw-v1/validation-00000-of-(…): downloading bytes:           |  0.00B            

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/1801350 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

Train rows: 1801350
Validation rows: 3760


In [9]:
def build_prompt_tensor(
    dataset,
    tokenizer,
    num_prompts,
    prefix_length,
    stride,
    token_offset,
):
    required_tokens = (
        token_offset
        + (num_prompts - 1) * stride
        + prefix_length
    )

    collected_token_ids = []

    eos_token_id = tokenizer.eos_token_id

    for row in dataset:
        text = row["text"].strip()

        if len(text) == 0:
            continue

        row_token_ids = tokenizer.encode(
            text,
            add_special_tokens=False,
        )

        if len(row_token_ids) == 0:
            continue

        collected_token_ids.extend(
            row_token_ids
        )

        if eos_token_id is not None:
            collected_token_ids.append(
                eos_token_id
            )

        if len(collected_token_ids) >= required_tokens:
            break

    assert len(collected_token_ids) >= required_tokens

    token_stream = torch.tensor(
        collected_token_ids,
        dtype=torch.long,
    )

    prompts = torch.stack([
        token_stream[
            token_offset + index * stride:
            token_offset + index * stride
            + prefix_length
        ]
        for index in range(num_prompts)
    ])

    assert prompts.shape == (
        num_prompts,
        prefix_length,
    )

    return prompts

In [10]:
PREFIX_LENGTH = 32
PROMPT_STRIDE = 64

NUM_TRAIN_PROMPTS = 320
NUM_VALIDATION_PROMPTS = 80

train_prompt_tokens = build_prompt_tensor(
    dataset=wikitext_train,
    tokenizer=tokenizer,
    num_prompts=NUM_TRAIN_PROMPTS,
    prefix_length=PREFIX_LENGTH,
    stride=PROMPT_STRIDE,
    token_offset=4096,
)

validation_prompt_tokens = build_prompt_tensor(
    dataset=wikitext_validation,
    tokenizer=tokenizer,
    num_prompts=NUM_VALIDATION_PROMPTS,
    prefix_length=PREFIX_LENGTH,
    stride=PROMPT_STRIDE,
    token_offset=1024,
)

saved_test_prompts = torch.load(
    TEST_PROMPT_PATH,
    map_location="cpu",
    weights_only=False,
)

test_prompt_tokens = saved_test_prompts[
    "token_ids"
].long()

print(
    "Train prompts:",
    train_prompt_tokens.shape,
)

print(
    "Validation prompts:",
    validation_prompt_tokens.shape,
)

print(
    "Test prompts:",
    test_prompt_tokens.shape,
)

assert train_prompt_tokens.shape == (
    320,
    32,
)

assert validation_prompt_tokens.shape == (
    80,
    32,
)

assert test_prompt_tokens.shape == (
    80,
    32,
)

Train prompts: torch.Size([320, 32])
Validation prompts: torch.Size([80, 32])
Test prompts: torch.Size([80, 32])


In [11]:
for split_name, prompts in [
    ("TRAIN", train_prompt_tokens),
    ("VALIDATION", validation_prompt_tokens),
    ("TEST", test_prompt_tokens),
]:
    print("\n" + "=" * 80)
    print(split_name)

    for index in range(2):
        decoded = tokenizer.decode(
            prompts[index].tolist()
        )

        print(f"\nPrompt {index}:")
        print(repr(decoded))


TRAIN

Prompt 0:
' the same timeline as the original game and its television anime adaptation , the cast of Valkyria Chronicles could make appearances , which pleased the team . The opening theme'

Prompt 1:
' singer Faylan . The ending theme , " Someday the Flowers of Light Will Bloom " ( いつか咲く光の花'

VALIDATION

Prompt 0:
'The species can be divided into four genetically distinct populations , one widespread population , and three which have diverged due to small effective population sizes , possibly due to adaptation'

Prompt 1:
' lobster " . The populations in the Mediterranean Sea are distinct from those in the Atlantic Ocean . The last distinct population is found in the Netherlands : samples from the O'

TEST

Prompt 0:
" .<|endoftext|>His father died around 740 . Du Fu would have been allowed to enter the civil service because of his father 's rank , but he is thought"

Prompt 1:
' in domestic affairs .<|endoftext|>In the autumn of 744 , he met Li Bai ( Li Po ) for the first tim

In [12]:
def prompt_rows_as_tuples(prompt_tensor):
    return {
        tuple(row.tolist())
        for row in prompt_tensor
    }


train_prompt_set = prompt_rows_as_tuples(
    train_prompt_tokens
)

validation_prompt_set = prompt_rows_as_tuples(
    validation_prompt_tokens
)

test_prompt_set = prompt_rows_as_tuples(
    test_prompt_tokens
)

train_validation_overlap = len(
    train_prompt_set & validation_prompt_set
)

train_test_overlap = len(
    train_prompt_set & test_prompt_set
)

validation_test_overlap = len(
    validation_prompt_set & test_prompt_set
)

print(
    "Train-validation exact overlap:",
    train_validation_overlap,
)

print(
    "Train-test exact overlap:",
    train_test_overlap,
)

print(
    "Validation-test exact overlap:",
    validation_test_overlap,
)

assert train_validation_overlap == 0
assert train_test_overlap == 0
assert validation_test_overlap == 0

print("No exact prompt overlap found.")

Train-validation exact overlap: 0
Train-test exact overlap: 0
Validation-test exact overlap: 0
No exact prompt overlap found.


In [13]:
PROMPT_SPLIT_PATH = (
    TRAINING_DATA_DIR
    / "notebook19_prompt_splits_seed42.pt"
)

torch.save(
    {
        "train_prompt_tokens":
            train_prompt_tokens,

        "validation_prompt_tokens":
            validation_prompt_tokens,

        "test_prompt_tokens":
            test_prompt_tokens,

        "training_dataset":
            "wikitext-103-raw-v1/train",

        "validation_dataset":
            "wikitext-103-raw-v1/validation",

        "test_dataset":
            "wikitext-103-raw-v1/test",

        "prefix_length":
            PREFIX_LENGTH,

        "prompt_stride":
            PROMPT_STRIDE,

        "seed":
            SEED,
    },
    PROMPT_SPLIT_PATH,
)

print("Saved prompt splits:")
print(PROMPT_SPLIT_PATH)

print(
    "File size MB:",
    PROMPT_SPLIT_PATH.stat().st_size
    / (1024 ** 2),
)

Saved prompt splits:
/content/drive/MyDrive/ma2288_nextlat/data_v3/notebook19_prompt_splits_seed42.pt
File size MB: 0.11974430084228516


In [14]:
print("=" * 80)
print("NOTEBOOK 19 — PHASE A SUMMARY")
print("=" * 80)

print("Device:", device)
print("Target model:", MODEL_NAME)
print("Hidden size:", HIDDEN_SIZE)
print("Vocabulary size:", VOCAB_SIZE)

print("\nBase checkpoint:")
print(BASE_CHECKPOINT_PATH)

print("\nPrompt shapes:")
print(
    "Train:",
    tuple(train_prompt_tokens.shape),
)
print(
    "Validation:",
    tuple(validation_prompt_tokens.shape),
)
print(
    "Test:",
    tuple(test_prompt_tokens.shape),
)

print("\nPrompt overlap:")
print(
    "Train-validation:",
    train_validation_overlap,
)
print(
    "Train-test:",
    train_test_overlap,
)
print(
    "Validation-test:",
    validation_test_overlap,
)

print("\nSaved split:")
print(PROMPT_SPLIT_PATH)

print("\nPHASE A COMPLETE")

NOTEBOOK 19 — PHASE A SUMMARY
Device: cuda
Target model: distilgpt2
Hidden size: 768
Vocabulary size: 50257

Base checkpoint:
/content/drive/MyDrive/ma2288_nextlat/checkpoints/multistep_transition_seed42.pt

Prompt shapes:
Train: (320, 32)
Validation: (80, 32)
Test: (80, 32)

Prompt overlap:
Train-validation: 0
Train-test: 0
Validation-test: 0

Saved split:
/content/drive/MyDrive/ma2288_nextlat/data_v3/notebook19_prompt_splits_seed42.pt

PHASE A COMPLETE


In [15]:
prompt_split_data = torch.load(
    PROMPT_SPLIT_PATH,
    map_location="cpu",
    weights_only=False,
)

train_prompt_tokens = prompt_split_data[
    "train_prompt_tokens"
].long()

validation_prompt_tokens = prompt_split_data[
    "validation_prompt_tokens"
].long()

test_prompt_tokens = prompt_split_data[
    "test_prompt_tokens"
].long()

print("Train:", train_prompt_tokens.shape)
print("Validation:", validation_prompt_tokens.shape)
print("Test:", test_prompt_tokens.shape)

Train: torch.Size([320, 32])
Validation: torch.Size([80, 32])
Test: torch.Size([80, 32])


In [16]:
class PromptDataset(Dataset):
    def __init__(self, prompt_tokens):
        self.prompt_tokens = prompt_tokens

    def __len__(self):
        return self.prompt_tokens.shape[0]

    def __getitem__(self, index):
        return self.prompt_tokens[index]


train_dataset = PromptDataset(
    train_prompt_tokens
)

validation_dataset = PromptDataset(
    validation_prompt_tokens
)

TRAIN_BATCH_SIZE = 4
VALIDATION_BATCH_SIZE = 4

train_loader = DataLoader(
    train_dataset,
    batch_size=TRAIN_BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=VALIDATION_BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)

example_batch = next(iter(train_loader))

print("Example batch:", example_batch.shape)
print("dtype:", example_batch.dtype)

Example batch: torch.Size([4, 32])
dtype: torch.int64


In [17]:
@torch.no_grad()
def get_target_state_and_logits(input_ids):
    transformer_output = target_model.transformer(
        input_ids=input_ids,
        attention_mask=torch.ones_like(input_ids),
        return_dict=True,
    )

    last_hidden = (
        transformer_output
        .last_hidden_state[:, -1, :]
        .float()
    )

    last_logits = target_model.lm_head(
        last_hidden
    ).float()

    return last_hidden, last_logits

In [18]:
VALID_OBJECTIVES = [
    "decoder_kl",
    "local_lk",
    "survival_lk",
]


def compute_onpolicy_training_loss(
    transition_model,
    prompt_batch,
    objective_name,
    rollout_length=8,
    latent_weight=1.0,
    objective_weight=0.2,
    sampling_seed=42,
    epsilon=1e-8,
):
    assert objective_name in VALID_OBJECTIVES

    current_context = prompt_batch.to(
        device=device,
        dtype=torch.long,
        non_blocking=True,
    )

    batch_size = current_context.shape[0]

    target_hidden, target_logits = (
        get_target_state_and_logits(
            current_context
        )
    )

    # The initial draft state is the exact target state.
    draft_hidden = target_hidden.detach().clone()

    acceptance_by_step = []
    kl_by_step = []
    latent_losses = []
    normalized_l2_by_transition = []

    sampling_generator = torch.Generator(
        device=device.type
    )

    sampling_generator.manual_seed(
        sampling_seed
    )

    for step in range(rollout_length):
        # Frozen LM head still allows gradients to flow
        # from logits into draft_hidden.
        draft_logits = target_model.lm_head(
            draft_hidden
        ).float()

        target_log_probability = F.log_softmax(
            target_logits.detach(),
            dim=-1,
        )

        target_probability = (
            target_log_probability.exp()
        )

        draft_log_probability = F.log_softmax(
            draft_logits,
            dim=-1,
        )

        draft_probability = (
            draft_log_probability.exp()
        )

        # Exact expected speculative acceptance:
        # sum_x min(p(x), q(x)) = 1 - TV(p, q).
        acceptance_probability = torch.minimum(
            target_probability,
            draft_probability,
        ).sum(dim=-1)

        acceptance_probability = (
            acceptance_probability.clamp(
                min=epsilon,
                max=1.0,
            )
        )

        output_kl = torch.sum(
            target_probability
            * (
                target_log_probability
                - draft_log_probability
            ),
            dim=-1,
        )

        acceptance_by_step.append(
            acceptance_probability
        )

        kl_by_step.append(
            output_kl
        )

        # No transition is needed after the final
        # distribution has been evaluated.
        if step == rollout_length - 1:
            continue

        sampled_token_ids = torch.multinomial(
            draft_probability.detach(),
            num_samples=1,
            replacement=True,
            generator=sampling_generator,
        ).squeeze(1)

        token_embeddings = token_embedding(
            sampled_token_ids
        ).float()

        next_draft_hidden = transition_model(
            draft_hidden,
            token_embeddings,
        )

        current_context = torch.cat(
            [
                current_context,
                sampled_token_ids.unsqueeze(1),
            ],
            dim=1,
        )

        next_target_hidden, next_target_logits = (
            get_target_state_and_logits(
                current_context
            )
        )

        latent_difference = (
            next_draft_hidden
            - next_target_hidden.detach()
        )

        transition_latent_loss = (
            latent_difference
            .pow(2)
            .mean(dim=-1)
        )

        normalized_l2 = (
            latent_difference.norm(dim=-1)
            / next_target_hidden
            .detach()
            .norm(dim=-1)
            .clamp_min(epsilon)
        )

        latent_losses.append(
            transition_latent_loss
        )

        normalized_l2_by_transition.append(
            normalized_l2
        )

        draft_hidden = next_draft_hidden
        target_hidden = next_target_hidden
        target_logits = next_target_logits

    acceptance_tensor = torch.stack(
        acceptance_by_step,
        dim=1,
    )

    kl_tensor = torch.stack(
        kl_by_step,
        dim=1,
    )

    latent_tensor = torch.stack(
        latent_losses,
        dim=1,
    )

    normalized_l2_tensor = torch.stack(
        normalized_l2_by_transition,
        dim=1,
    )

    # Step 1 is exactly aligned because both models
    # begin from the same target hidden state.
    trainable_acceptance = (
        acceptance_tensor[:, 1:]
    )

    trainable_kl = kl_tensor[:, 1:]

    mean_latent_loss = latent_tensor.mean()

    decoder_kl_loss = trainable_kl.mean()

    local_lk_loss = (
        1.0 - trainable_acceptance
    ).mean()

    survival_probabilities = torch.cumprod(
        trainable_acceptance,
        dim=1,
    )

    normalized_expected_survival = (
        survival_probabilities.mean(dim=1)
    )

    survival_lk_loss = (
        1.0
        - normalized_expected_survival
    ).mean()

    if objective_name == "decoder_kl":
        selected_objective_loss = (
            decoder_kl_loss
        )

    elif objective_name == "local_lk":
        selected_objective_loss = (
            local_lk_loss
        )

    elif objective_name == "survival_lk":
        selected_objective_loss = (
            survival_lk_loss
        )

    total_loss = (
        latent_weight * mean_latent_loss
        + objective_weight
        * selected_objective_loss
    )

    return {
        "loss": total_loss,
        "latent_loss": mean_latent_loss,
        "selected_objective_loss":
            selected_objective_loss,
        "decoder_kl_loss":
            decoder_kl_loss,
        "local_lk_loss":
            local_lk_loss,
        "survival_lk_loss":
            survival_lk_loss,
        "acceptance_by_step":
            acceptance_tensor.mean(dim=0),
        "kl_by_step":
            kl_tensor.mean(dim=0),
        "normalized_l2_by_transition":
            normalized_l2_tensor.mean(dim=0),
        "survival_by_trainable_step":
            survival_probabilities.mean(dim=0),
        "expected_accepted_beyond_first":
            survival_probabilities
            .sum(dim=1)
            .mean(),
    }

In [19]:
sanity_prompt_batch = (
    train_prompt_tokens[:2]
)

sanity_model = ResidualLatentTransition(
    hidden_size=HIDDEN_SIZE,
    token_embedding_size=HIDDEN_SIZE,
    intermediate_size=512,
).to(device)

sanity_model.load_state_dict(
    base_state_dict,
    strict=True,
)

sanity_model.train()

sanity_metrics = (
    compute_onpolicy_training_loss(
        transition_model=sanity_model,
        prompt_batch=sanity_prompt_batch,
        objective_name="survival_lk",
        rollout_length=8,
        latent_weight=1.0,
        objective_weight=0.2,
        sampling_seed=SEED,
    )
)

for key in [
    "loss",
    "latent_loss",
    "selected_objective_loss",
    "decoder_kl_loss",
    "local_lk_loss",
    "survival_lk_loss",
    "expected_accepted_beyond_first",
]:
    print(
        f"{key}: "
        f"{sanity_metrics[key].item():.6f}"
    )

print("\nAcceptance by step:")
print(
    sanity_metrics[
        "acceptance_by_step"
    ].detach().cpu()
)

print("\nSurvival by trainable step:")
print(
    sanity_metrics[
        "survival_by_trainable_step"
    ].detach().cpu()
)

print("\nKL by step:")
print(
    sanity_metrics[
        "kl_by_step"
    ].detach().cpu()
)

print("\nNormalized L2 by transition:")
print(
    sanity_metrics[
        "normalized_l2_by_transition"
    ].detach().cpu()
)

loss: 0.715748
latent_loss: 0.541955
selected_objective_loss: 0.868962
decoder_kl_loss: 1.854573
local_lk_loss: 0.645723
survival_lk_loss: 0.868962
expected_accepted_beyond_first: 0.917269

Acceptance by step:
tensor([1.0000, 0.6053, 0.3406, 0.3810, 0.3331, 0.1385, 0.2502, 0.4312])

Survival by trainable step:
tensor([6.0533e-01, 2.0645e-01, 7.5616e-02, 2.6307e-02, 2.3241e-03, 8.9269e-04,
        3.5422e-04])

KL by step:
tensor([0.0000, 0.8007, 1.6828, 1.4794, 1.8324, 3.5448, 2.5332, 1.1088])

Normalized L2 by transition:
tensor([0.1010, 0.1438, 0.1201, 0.1588, 0.1259, 0.1508, 0.1026])


In [20]:
acceptance_values = (
    sanity_metrics[
        "acceptance_by_step"
    ]
    .detach()
    .cpu()
    .numpy()
)

survival_values = (
    sanity_metrics[
        "survival_by_trainable_step"
    ]
    .detach()
    .cpu()
    .numpy()
)

kl_values = (
    sanity_metrics[
        "kl_by_step"
    ]
    .detach()
    .cpu()
    .numpy()
)

assert np.isfinite(
    sanity_metrics["loss"].item()
)

assert np.all(
    acceptance_values >= 0
)

assert np.all(
    acceptance_values <= 1 + 1e-5
)

assert acceptance_values[0] > 0.999

assert kl_values[0] < 1e-4

assert np.all(
    survival_values[1:]
    <= survival_values[:-1] + 1e-5
)

print("Numerical sanity checks passed.")

Numerical sanity checks passed.


In [21]:
def compute_gradient_vector(
    objective_name,
    latent_weight,
    objective_weight,
):
    model = ResidualLatentTransition(
        hidden_size=HIDDEN_SIZE,
        token_embedding_size=HIDDEN_SIZE,
        intermediate_size=512,
    ).to(device)

    model.load_state_dict(
        base_state_dict,
        strict=True,
    )

    model.train()
    model.zero_grad(set_to_none=True)

    metrics = compute_onpolicy_training_loss(
        transition_model=model,
        prompt_batch=sanity_prompt_batch,
        objective_name=objective_name,
        rollout_length=8,
        latent_weight=latent_weight,
        objective_weight=objective_weight,
        sampling_seed=SEED,
    )

    metrics["loss"].backward()

    gradient_parts = []

    for parameter in model.parameters():
        if parameter.grad is not None:
            gradient_parts.append(
                parameter.grad
                .detach()
                .flatten()
                .cpu()
            )

    gradient_vector = torch.cat(
        gradient_parts
    )

    result = {
        "objective": objective_name,
        "loss": metrics["loss"].item(),
        "gradient_norm":
            gradient_vector.norm().item(),
        "gradient_vector":
            gradient_vector,
    }

    del model
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return result

In [22]:
objective_only_gradients = {}

for objective_name in VALID_OBJECTIVES:
    print("Computing gradient for:",
          objective_name)

    result = compute_gradient_vector(
        objective_name=objective_name,
        latent_weight=0.0,
        objective_weight=1.0,
    )

    objective_only_gradients[
        objective_name
    ] = result

    print(
        f"  loss={result['loss']:.6f}, "
        f"gradient_norm="
        f"{result['gradient_norm']:.6f}"
    )

Computing gradient for: decoder_kl
  loss=1.854573, gradient_norm=24.703747
Computing gradient for: local_lk
  loss=0.645723, gradient_norm=2.992851
Computing gradient for: survival_lk
  loss=0.868962, gradient_norm=1.928637


In [23]:
def gradient_cosine(name_a, name_b):
    gradient_a = (
        objective_only_gradients[
            name_a
        ]["gradient_vector"]
    )

    gradient_b = (
        objective_only_gradients[
            name_b
        ]["gradient_vector"]
    )

    return F.cosine_similarity(
        gradient_a.unsqueeze(0),
        gradient_b.unsqueeze(0),
    ).item()


gradient_comparison = pd.DataFrame([
    {
        "objective_a": "decoder_kl",
        "objective_b": "local_lk",
        "gradient_cosine":
            gradient_cosine(
                "decoder_kl",
                "local_lk",
            ),
    },
    {
        "objective_a": "decoder_kl",
        "objective_b": "survival_lk",
        "gradient_cosine":
            gradient_cosine(
                "decoder_kl",
                "survival_lk",
            ),
    },
    {
        "objective_a": "local_lk",
        "objective_b": "survival_lk",
        "gradient_cosine":
            gradient_cosine(
                "local_lk",
                "survival_lk",
            ),
    },
])

print(
    gradient_comparison.to_string(
        index=False
    )
)

objective_a objective_b  gradient_cosine
 decoder_kl    local_lk         0.742745
 decoder_kl survival_lk        -0.122426
   local_lk survival_lk         0.297054


In [24]:
combined_gradient_results = []

for objective_name in VALID_OBJECTIVES:
    result = compute_gradient_vector(
        objective_name=objective_name,
        latent_weight=1.0,
        objective_weight=0.2,
    )

    combined_gradient_results.append({
        "objective": objective_name,
        "combined_loss": result["loss"],
        "combined_gradient_norm":
            result["gradient_norm"],
    })

combined_gradient_df = pd.DataFrame(
    combined_gradient_results
)

print(
    combined_gradient_df.to_string(
        index=False
    )
)

  objective  combined_loss  combined_gradient_norm
 decoder_kl       0.912870                9.135363
   local_lk       0.671100                7.957505
survival_lk       0.715748                8.014733


In [25]:
pd.set_option(
    "display.max_columns",
    None,
)

pd.set_option(
    "display.width",
    200,
)

print("=" * 90)
print("NOTEBOOK 19 — PHASE B RESULTS")
print("=" * 90)

print("\nFORWARD METRICS")

for key in [
    "loss",
    "latent_loss",
    "decoder_kl_loss",
    "local_lk_loss",
    "survival_lk_loss",
    "expected_accepted_beyond_first",
]:
    print(
        f"{key}: "
        f"{sanity_metrics[key].item():.6f}"
    )

print("\nACCEPTANCE BY STEP")

print(
    sanity_metrics[
        "acceptance_by_step"
    ].detach().cpu()
)

print("\nSURVIVAL BY TRAINABLE STEP")

print(
    sanity_metrics[
        "survival_by_trainable_step"
    ].detach().cpu()
)

print("\nOBJECTIVE-ONLY GRADIENTS")

for objective_name in VALID_OBJECTIVES:
    result = objective_only_gradients[
        objective_name
    ]

    print(
        f"{objective_name}: "
        f"loss={result['loss']:.6f}, "
        f"gradient_norm="
        f"{result['gradient_norm']:.6f}"
    )

print("\nGRADIENT COSINE")

print(
    gradient_comparison.to_string(
        index=False
    )
)

print("\nCOMBINED GRADIENTS")

print(
    combined_gradient_df.to_string(
        index=False
    )
)

print("\nPHASE B COMPLETE")

NOTEBOOK 19 — PHASE B RESULTS

FORWARD METRICS
loss: 0.715748
latent_loss: 0.541955
decoder_kl_loss: 1.854573
local_lk_loss: 0.645723
survival_lk_loss: 0.868962
expected_accepted_beyond_first: 0.917269

ACCEPTANCE BY STEP
tensor([1.0000, 0.6053, 0.3406, 0.3810, 0.3331, 0.1385, 0.2502, 0.4312])

SURVIVAL BY TRAINABLE STEP
tensor([6.0533e-01, 2.0645e-01, 7.5616e-02, 2.6307e-02, 2.3241e-03, 8.9269e-04,
        3.5422e-04])

OBJECTIVE-ONLY GRADIENTS
decoder_kl: loss=1.854573, gradient_norm=24.703747
local_lk: loss=0.645723, gradient_norm=2.992851
survival_lk: loss=0.868962, gradient_norm=1.928637

GRADIENT COSINE
objective_a objective_b  gradient_cosine
 decoder_kl    local_lk         0.742745
 decoder_kl survival_lk        -0.122426
   local_lk survival_lk         0.297054

COMBINED GRADIENTS
  objective  combined_loss  combined_gradient_norm
 decoder_kl       0.912870                9.135363
   local_lk       0.671100                7.957505
survival_lk       0.715748                8.01

In [26]:
latent_only_result = compute_gradient_vector(
    objective_name="decoder_kl",
    latent_weight=1.0,
    objective_weight=0.0,
)

LATENT_GRADIENT_NORM = (
    latent_only_result[
        "gradient_norm"
    ]
)

print(
    "Latent-only loss:",
    latent_only_result["loss"],
)

print(
    "Latent-only gradient norm:",
    LATENT_GRADIENT_NORM,
)

Latent-only loss: 0.5419554114341736
Latent-only gradient norm: 7.993977069854736


In [27]:
TARGET_OBJECTIVE_TO_LATENT_RATIO = 0.25

calibrated_objective_weights = {}

for objective_name in VALID_OBJECTIVES:
    objective_gradient_norm = (
        objective_only_gradients[
            objective_name
        ]["gradient_norm"]
    )

    calibrated_weight = (
        TARGET_OBJECTIVE_TO_LATENT_RATIO
        * LATENT_GRADIENT_NORM
        / objective_gradient_norm
    )

    calibrated_objective_weights[
        objective_name
    ] = calibrated_weight

print("Calibrated objective weights:\n")

for objective_name, weight in (
    calibrated_objective_weights.items()
):
    effective_gradient = (
        weight
        * objective_only_gradients[
            objective_name
        ]["gradient_norm"]
    )

    print(
        f"{objective_name:12s} "
        f"weight={weight:.6f} | "
        f"effective_gradient="
        f"{effective_gradient:.6f}"
    )

print(
    "\nTarget effective gradient:",
    TARGET_OBJECTIVE_TO_LATENT_RATIO
    * LATENT_GRADIENT_NORM,
)

Calibrated objective weights:

decoder_kl   weight=0.080898 | effective_gradient=1.998494
local_lk     weight=0.667756 | effective_gradient=1.998494
survival_lk  weight=1.036221 | effective_gradient=1.998494

Target effective gradient: 1.998494267463684


In [28]:
calibrated_combined_results = {}
calibrated_combined_rows = []

for objective_name in VALID_OBJECTIVES:
    calibrated_weight = (
        calibrated_objective_weights[
            objective_name
        ]
    )

    result = compute_gradient_vector(
        objective_name=objective_name,
        latent_weight=1.0,
        objective_weight=calibrated_weight,
    )

    calibrated_combined_results[
        objective_name
    ] = result

    calibrated_combined_rows.append({
        "objective": objective_name,
        "objective_weight":
            calibrated_weight,
        "combined_loss":
            result["loss"],
        "combined_gradient_norm":
            result["gradient_norm"],
    })

calibrated_combined_df = pd.DataFrame(
    calibrated_combined_rows
)

print(
    calibrated_combined_df.to_string(
        index=False
    )
)

  objective  objective_weight  combined_loss  combined_gradient_norm
 decoder_kl          0.080898       0.691987                8.119906
   local_lk          0.667756       0.973141                8.047327
survival_lk          1.036221       1.442392                8.297541


In [29]:
latent_gradient_vector = (
    latent_only_result[
        "gradient_vector"
    ]
)

calibrated_direction_rows = []

for objective_name in VALID_OBJECTIVES:
    combined_gradient_vector = (
        calibrated_combined_results[
            objective_name
        ]["gradient_vector"]
    )

    cosine_with_latent = F.cosine_similarity(
        combined_gradient_vector.unsqueeze(0),
        latent_gradient_vector.unsqueeze(0),
    ).item()

    relative_change = (
        (
            combined_gradient_vector
            - latent_gradient_vector
        ).norm()
        / latent_gradient_vector.norm()
    ).item()

    calibrated_direction_rows.append({
        "objective": objective_name,
        "cosine_with_latent_gradient":
            cosine_with_latent,
        "relative_gradient_change":
            relative_change,
    })

calibrated_direction_df = pd.DataFrame(
    calibrated_direction_rows
)

print(
    calibrated_direction_df.to_string(
        index=False
    )
)

  objective  cosine_with_latent_gradient  relative_gradient_change
 decoder_kl                     0.969606                  0.250000
   local_lk                     0.969225                  0.250003
survival_lk                     0.970837                  0.250000


In [30]:
CALIBRATION_PATH = (
    TRAINING_DATA_DIR
    / "notebook19_gradient_calibration_seed42.json"
)

calibration_to_save = {
    "seed": SEED,
    "target_objective_to_latent_ratio":
        TARGET_OBJECTIVE_TO_LATENT_RATIO,
    "latent_gradient_norm":
        LATENT_GRADIENT_NORM,
    "objective_gradient_norms": {
        objective_name:
            objective_only_gradients[
                objective_name
            ]["gradient_norm"]
        for objective_name
        in VALID_OBJECTIVES
    },
    "calibrated_objective_weights":
        calibrated_objective_weights,
}

with open(
    CALIBRATION_PATH,
    "w",
) as file:
    json.dump(
        calibration_to_save,
        file,
        indent=2,
    )

print("Saved calibration:")
print(CALIBRATION_PATH)

Saved calibration:
/content/drive/MyDrive/ma2288_nextlat/data_v3/notebook19_gradient_calibration_seed42.json


In [31]:
print("=" * 90)
print("NOTEBOOK 19 — PHASE B2 CALIBRATION")
print("=" * 90)

print(
    "\nLatent gradient norm:",
    LATENT_GRADIENT_NORM,
)

print("\nCalibrated weights:")

for objective_name, weight in (
    calibrated_objective_weights.items()
):
    print(
        f"{objective_name}: "
        f"{weight:.8f}"
    )

print("\nCalibrated combined gradients:")

print(
    calibrated_combined_df.to_string(
        index=False
    )
)

print("\nDirection relative to latent gradient:")

print(
    calibrated_direction_df.to_string(
        index=False
    )
)

print("\nPHASE B2 COMPLETE")

NOTEBOOK 19 — PHASE B2 CALIBRATION

Latent gradient norm: 7.993977069854736

Calibrated weights:
decoder_kl: 0.08089843
local_lk: 0.66775612
survival_lk: 1.03622130

Calibrated combined gradients:
  objective  objective_weight  combined_loss  combined_gradient_norm
 decoder_kl          0.080898       0.691987                8.119906
   local_lk          0.667756       0.973141                8.047327
survival_lk          1.036221       1.442392                8.297541

Direction relative to latent gradient:
  objective  cosine_with_latent_gradient  relative_gradient_change
 decoder_kl                     0.969606                  0.250000
   local_lk                     0.969225                  0.250003
survival_lk                     0.970837                  0.250000

PHASE B2 COMPLETE


In [32]:
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4

MAX_EPOCHS = 8
PATIENCE = 2
GRADIENT_CLIP = 1.0

ROLLOUT_LENGTH = 8

print("Training configuration:")
print("Learning rate:", LEARNING_RATE)
print("Weight decay:", WEIGHT_DECAY)
print("Maximum epochs:", MAX_EPOCHS)
print("Patience:", PATIENCE)
print("Rollout length:", ROLLOUT_LENGTH)

print("\nObjective weights:")

for objective_name, weight in (
    calibrated_objective_weights.items()
):
    print(
        f"{objective_name}: {weight:.8f}"
    )

Training configuration:
Learning rate: 0.0001
Weight decay: 0.0001
Maximum epochs: 8
Patience: 2
Rollout length: 8

Objective weights:
decoder_kl: 0.08089843
local_lk: 0.66775612
survival_lk: 1.03622130


In [33]:
def create_train_loader(seed):
    data_generator = torch.Generator()
    data_generator.manual_seed(seed)

    return DataLoader(
        train_dataset,
        batch_size=TRAIN_BATCH_SIZE,
        shuffle=True,
        num_workers=0,
        pin_memory=torch.cuda.is_available(),
        generator=data_generator,
    )


fixed_validation_loader = DataLoader(
    validation_dataset,
    batch_size=VALIDATION_BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)

In [34]:
@torch.no_grad()
def evaluate_training_objective(
    transition_model,
    objective_name,
    objective_weight,
    data_loader,
    validation_seed=42000,
):
    transition_model.eval()

    total_samples = 0

    scalar_sums = {
        "loss": 0.0,
        "latent_loss": 0.0,
        "selected_objective_loss": 0.0,
        "decoder_kl_loss": 0.0,
        "local_lk_loss": 0.0,
        "survival_lk_loss": 0.0,
        "expected_accepted_beyond_first": 0.0,
    }

    acceptance_sum = None
    survival_sum = None
    kl_sum = None
    normalized_l2_sum = None

    for batch_index, prompt_batch in enumerate(
        data_loader
    ):
        batch_size = prompt_batch.shape[0]

        metrics = compute_onpolicy_training_loss(
            transition_model=transition_model,
            prompt_batch=prompt_batch,
            objective_name=objective_name,
            rollout_length=ROLLOUT_LENGTH,
            latent_weight=1.0,
            objective_weight=objective_weight,
            sampling_seed=(
                validation_seed + batch_index
            ),
        )

        total_samples += batch_size

        for key in scalar_sums:
            scalar_sums[key] += (
                metrics[key].item()
                * batch_size
            )

        current_acceptance = (
            metrics["acceptance_by_step"]
            .detach()
            .cpu()
            * batch_size
        )

        current_survival = (
            metrics[
                "survival_by_trainable_step"
            ]
            .detach()
            .cpu()
            * batch_size
        )

        current_kl = (
            metrics["kl_by_step"]
            .detach()
            .cpu()
            * batch_size
        )

        current_l2 = (
            metrics[
                "normalized_l2_by_transition"
            ]
            .detach()
            .cpu()
            * batch_size
        )

        if acceptance_sum is None:
            acceptance_sum = current_acceptance
            survival_sum = current_survival
            kl_sum = current_kl
            normalized_l2_sum = current_l2
        else:
            acceptance_sum += current_acceptance
            survival_sum += current_survival
            kl_sum += current_kl
            normalized_l2_sum += current_l2

    result = {
        key: value / total_samples
        for key, value in scalar_sums.items()
    }

    result["acceptance_by_step"] = (
        acceptance_sum / total_samples
    ).numpy()

    result[
        "survival_by_trainable_step"
    ] = (
        survival_sum / total_samples
    ).numpy()

    result["kl_by_step"] = (
        kl_sum / total_samples
    ).numpy()

    result[
        "normalized_l2_by_transition"
    ] = (
        normalized_l2_sum / total_samples
    ).numpy()

    return result

In [35]:
def clone_state_dict_to_cpu(model):
    return {
        key: value.detach().cpu().clone()
        for key, value
        in model.state_dict().items()
    }


def train_one_objective(
    objective_name,
    objective_weight,
):
    print("\n" + "=" * 90)
    print("TRAINING:", objective_name)
    print("Objective weight:", objective_weight)
    print("=" * 90)

    model = ResidualLatentTransition(
        hidden_size=HIDDEN_SIZE,
        token_embedding_size=HIDDEN_SIZE,
        intermediate_size=512,
    ).to(device)

    model.load_state_dict(
        base_state_dict,
        strict=True,
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )

    baseline_validation = (
        evaluate_training_objective(
            transition_model=model,
            objective_name=objective_name,
            objective_weight=objective_weight,
            data_loader=fixed_validation_loader,
            validation_seed=42000,
        )
    )

    print("\nBaseline validation:")
    print(
        f"loss="
        f"{baseline_validation['loss']:.6f} | "
        f"latent="
        f"{baseline_validation['latent_loss']:.6f} | "
        f"KL="
        f"{baseline_validation['decoder_kl_loss']:.6f} | "
        f"local_LK="
        f"{baseline_validation['local_lk_loss']:.6f} | "
        f"survival_LK="
        f"{baseline_validation['survival_lk_loss']:.6f} | "
        f"expected_beyond_first="
        f"{baseline_validation['expected_accepted_beyond_first']:.6f}"
    )

    best_validation_loss = (
        baseline_validation["loss"]
    )

    best_epoch = 0
    best_state_dict = clone_state_dict_to_cpu(
        model
    )

    best_validation_metrics = copy.deepcopy(
        baseline_validation
    )

    epochs_without_improvement = 0
    history = []

    train_loader_for_model = create_train_loader(
        SEED
    )

    for epoch in range(
        1,
        MAX_EPOCHS + 1,
    ):
        model.train()

        train_samples = 0

        train_sums = {
            "loss": 0.0,
            "latent_loss": 0.0,
            "selected_objective_loss": 0.0,
            "decoder_kl_loss": 0.0,
            "local_lk_loss": 0.0,
            "survival_lk_loss": 0.0,
            "expected_accepted_beyond_first": 0.0,
        }

        for batch_index, prompt_batch in enumerate(
            train_loader_for_model
        ):
            optimizer.zero_grad(
                set_to_none=True
            )

            metrics = (
                compute_onpolicy_training_loss(
                    transition_model=model,
                    prompt_batch=prompt_batch,
                    objective_name=objective_name,
                    rollout_length=ROLLOUT_LENGTH,
                    latent_weight=1.0,
                    objective_weight=objective_weight,
                    sampling_seed=(
                        SEED
                        + epoch * 10000
                        + batch_index
                    ),
                )
            )

            loss = metrics["loss"]

            assert torch.isfinite(loss), (
                f"Non-finite loss for "
                f"{objective_name}"
            )

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                GRADIENT_CLIP,
            )

            optimizer.step()

            batch_size = prompt_batch.shape[0]
            train_samples += batch_size

            for key in train_sums:
                train_sums[key] += (
                    metrics[key].item()
                    * batch_size
                )

            del metrics, loss

        train_means = {
            key: value / train_samples
            for key, value
            in train_sums.items()
        }

        validation_metrics = (
            evaluate_training_objective(
                transition_model=model,
                objective_name=objective_name,
                objective_weight=objective_weight,
                data_loader=fixed_validation_loader,
                validation_seed=42000,
            )
        )

        history_row = {
            "objective": objective_name,
            "epoch": epoch,

            "train_loss":
                train_means["loss"],

            "train_latent_loss":
                train_means["latent_loss"],

            "train_selected_objective_loss":
                train_means[
                    "selected_objective_loss"
                ],

            "train_expected_beyond_first":
                train_means[
                    "expected_accepted_beyond_first"
                ],

            "validation_loss":
                validation_metrics["loss"],

            "validation_latent_loss":
                validation_metrics[
                    "latent_loss"
                ],

            "validation_decoder_kl":
                validation_metrics[
                    "decoder_kl_loss"
                ],

            "validation_local_lk":
                validation_metrics[
                    "local_lk_loss"
                ],

            "validation_survival_lk":
                validation_metrics[
                    "survival_lk_loss"
                ],

            "validation_expected_beyond_first":
                validation_metrics[
                    "expected_accepted_beyond_first"
                ],
        }

        history.append(history_row)

        print(
            f"Epoch {epoch:02d} | "
            f"train={train_means['loss']:.6f} | "
            f"val={validation_metrics['loss']:.6f} | "
            f"latent="
            f"{validation_metrics['latent_loss']:.6f} | "
            f"KL="
            f"{validation_metrics['decoder_kl_loss']:.6f} | "
            f"local_LK="
            f"{validation_metrics['local_lk_loss']:.6f} | "
            f"survival_LK="
            f"{validation_metrics['survival_lk_loss']:.6f} | "
            f"E[beyond1]="
            f"{validation_metrics['expected_accepted_beyond_first']:.6f}"
        )

        if (
            validation_metrics["loss"]
            < best_validation_loss - 1e-6
        ):
            best_validation_loss = (
                validation_metrics["loss"]
            )

            best_epoch = epoch

            best_state_dict = (
                clone_state_dict_to_cpu(
                    model
                )
            )

            best_validation_metrics = (
                copy.deepcopy(
                    validation_metrics
                )
            )

            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if (
            epochs_without_improvement
            >= PATIENCE
        ):
            print(
                f"Early stopping at "
                f"epoch {epoch}."
            )
            break

        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    model.load_state_dict(
        best_state_dict,
        strict=True,
    )

    print("\nBest epoch:", best_epoch)
    print(
        "Best validation loss:",
        best_validation_loss,
    )

    return {
        "model": model,
        "objective": objective_name,
        "objective_weight": objective_weight,
        "best_epoch": best_epoch,
        "best_validation_loss":
            best_validation_loss,
        "baseline_validation_metrics":
            baseline_validation,
        "best_validation_metrics":
            best_validation_metrics,
        "history": history,
    }

In [36]:
smoke_model = ResidualLatentTransition(
    hidden_size=HIDDEN_SIZE,
    token_embedding_size=HIDDEN_SIZE,
    intermediate_size=512,
).to(device)

smoke_model.load_state_dict(
    base_state_dict,
    strict=True,
)

smoke_optimizer = torch.optim.AdamW(
    smoke_model.parameters(),
    lr=LEARNING_RATE,
)

smoke_batch = train_prompt_tokens[:2]

parameter_before = (
    smoke_model
    .network[0]
    .weight
    .detach()
    .clone()
)

smoke_optimizer.zero_grad(
    set_to_none=True
)

smoke_metrics = (
    compute_onpolicy_training_loss(
        transition_model=smoke_model,
        prompt_batch=smoke_batch,
        objective_name="survival_lk",
        rollout_length=ROLLOUT_LENGTH,
        latent_weight=1.0,
        objective_weight=(
            calibrated_objective_weights[
                "survival_lk"
            ]
        ),
        sampling_seed=SEED,
    )
)

smoke_metrics["loss"].backward()

torch.nn.utils.clip_grad_norm_(
    smoke_model.parameters(),
    GRADIENT_CLIP,
)

smoke_optimizer.step()

parameter_after = (
    smoke_model
    .network[0]
    .weight
    .detach()
)

maximum_parameter_change = (
    parameter_after - parameter_before
).abs().max().item()

print(
    "Smoke-test loss:",
    smoke_metrics["loss"].item(),
)

print(
    "Maximum parameter change:",
    maximum_parameter_change,
)

assert maximum_parameter_change > 0
assert np.isfinite(
    maximum_parameter_change
)

print("Training smoke test passed.")

del smoke_model
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

Smoke-test loss: 1.4423918724060059
Maximum parameter change: 0.00010097026824951172
Training smoke test passed.


In [37]:
decoder_kl_result = train_one_objective(
    objective_name="decoder_kl",
    objective_weight=(
        calibrated_objective_weights[
            "decoder_kl"
        ]
    ),
)


TRAINING: decoder_kl
Objective weight: 0.08089842743268583

Baseline validation:
loss=1.361822 | latent=1.196028 | KL=2.049413 | local_LK=0.616946 | survival_LK=0.892898 | expected_beyond_first=0.749712
Epoch 01 | train=1.741355 | val=1.583972 | latent=1.424069 | KL=1.976596 | local_LK=0.620943 | survival_LK=0.895930 | E[beyond1]=0.728490
Epoch 02 | train=1.686771 | val=1.522693 | latent=1.368133 | KL=1.910543 | local_LK=0.620123 | survival_LK=0.895112 | E[beyond1]=0.734213
Early stopping at epoch 2.

Best epoch: 0
Best validation loss: 1.3618221759796143


In [38]:
decoder_kl_checkpoint_path = (
    CHECKPOINT_OUTPUT_DIR
    / "decoder_kl_transition_seed42.pt"
)

torch.save(
    {
        "model_state_dict":
            clone_state_dict_to_cpu(
                decoder_kl_result["model"]
            ),
        "objective": "decoder_kl",
        "objective_weight":
            decoder_kl_result[
                "objective_weight"
            ],
        "best_epoch":
            decoder_kl_result["best_epoch"],
        "best_validation_loss":
            decoder_kl_result[
                "best_validation_loss"
            ],
        "baseline_validation_metrics":
            decoder_kl_result[
                "baseline_validation_metrics"
            ],
        "best_validation_metrics":
            decoder_kl_result[
                "best_validation_metrics"
            ],
        "training_history":
            decoder_kl_result["history"],
        "initial_checkpoint":
            str(BASE_CHECKPOINT_PATH),
        "prompt_split":
            str(PROMPT_SPLIT_PATH),
        "seed": SEED,
    },
    decoder_kl_checkpoint_path,
)

print("Saved:")
print(decoder_kl_checkpoint_path)

Saved:
/content/drive/MyDrive/ma2288_nextlat/checkpoints_v3/decoder_kl_transition_seed42.pt


In [39]:
local_lk_result = train_one_objective(
    objective_name="local_lk",
    objective_weight=(
        calibrated_objective_weights[
            "local_lk"
        ]
    ),
)


TRAINING: local_lk
Objective weight: 0.6677561205854412

Baseline validation:
loss=1.607997 | latent=1.196028 | KL=2.049413 | local_LK=0.616946 | survival_LK=0.892898 | expected_beyond_first=0.749712
Epoch 01 | train=2.005444 | val=1.641107 | latent=1.236182 | KL=1.939673 | local_LK=0.606397 | survival_LK=0.893365 | E[beyond1]=0.746441
Epoch 02 | train=1.939401 | val=1.702437 | latent=1.290699 | KL=1.971311 | local_LK=0.616598 | survival_LK=0.894517 | E[beyond1]=0.738379
Early stopping at epoch 2.

Best epoch: 0
Best validation loss: 1.6079973995685577


In [40]:
local_lk_checkpoint_path = (
    CHECKPOINT_OUTPUT_DIR
    / "local_lk_transition_seed42.pt"
)

torch.save(
    {
        "model_state_dict":
            clone_state_dict_to_cpu(
                local_lk_result["model"]
            ),
        "objective": "local_lk",
        "objective_weight":
            local_lk_result[
                "objective_weight"
            ],
        "best_epoch":
            local_lk_result["best_epoch"],
        "best_validation_loss":
            local_lk_result[
                "best_validation_loss"
            ],
        "baseline_validation_metrics":
            local_lk_result[
                "baseline_validation_metrics"
            ],
        "best_validation_metrics":
            local_lk_result[
                "best_validation_metrics"
            ],
        "training_history":
            local_lk_result["history"],
        "initial_checkpoint":
            str(BASE_CHECKPOINT_PATH),
        "prompt_split":
            str(PROMPT_SPLIT_PATH),
        "seed": SEED,
    },
    local_lk_checkpoint_path,
)

print("Saved:")
print(local_lk_checkpoint_path)

Saved:
/content/drive/MyDrive/ma2288_nextlat/checkpoints_v3/local_lk_transition_seed42.pt


In [41]:
survival_lk_result = train_one_objective(
    objective_name="survival_lk",
    objective_weight=(
        calibrated_objective_weights[
            "survival_lk"
        ]
    ),
)


TRAINING: survival_lk
Objective weight: 1.036221296608552

Baseline validation:
loss=2.121268 | latent=1.196028 | KL=2.049413 | local_LK=0.616946 | survival_LK=0.892898 | expected_beyond_first=0.749712
Epoch 01 | train=2.536683 | val=2.215203 | latent=1.289551 | KL=1.957360 | local_LK=0.613412 | survival_LK=0.893296 | E[beyond1]=0.746930
Epoch 02 | train=2.539852 | val=2.118697 | latent=1.190946 | KL=2.041377 | local_LK=0.621471 | survival_LK=0.895321 | E[beyond1]=0.732754
Epoch 03 | train=2.251279 | val=2.300557 | latent=1.375012 | KL=1.969532 | local_LK=0.611322 | survival_LK=0.893192 | E[beyond1]=0.747653
Epoch 04 | train=2.279296 | val=2.314220 | latent=1.390362 | KL=1.990502 | local_LK=0.610409 | survival_LK=0.891564 | E[beyond1]=0.759053
Early stopping at epoch 4.

Best epoch: 2
Best validation loss: 2.1186966776847838


In [42]:
survival_lk_checkpoint_path = (
    CHECKPOINT_OUTPUT_DIR
    / "survival_lk_transition_seed42.pt"
)

torch.save(
    {
        "model_state_dict":
            clone_state_dict_to_cpu(
                survival_lk_result["model"]
            ),
        "objective": "survival_lk",
        "objective_weight":
            survival_lk_result[
                "objective_weight"
            ],
        "best_epoch":
            survival_lk_result["best_epoch"],
        "best_validation_loss":
            survival_lk_result[
                "best_validation_loss"
            ],
        "baseline_validation_metrics":
            survival_lk_result[
                "baseline_validation_metrics"
            ],
        "best_validation_metrics":
            survival_lk_result[
                "best_validation_metrics"
            ],
        "training_history":
            survival_lk_result["history"],
        "initial_checkpoint":
            str(BASE_CHECKPOINT_PATH),
        "prompt_split":
            str(PROMPT_SPLIT_PATH),
        "seed": SEED,
    },
    survival_lk_checkpoint_path,
)

print("Saved:")
print(survival_lk_checkpoint_path)

Saved:
/content/drive/MyDrive/ma2288_nextlat/checkpoints_v3/survival_lk_transition_seed42.pt


In [43]:
training_results = [
    decoder_kl_result,
    local_lk_result,
    survival_lk_result,
]

training_summary_rows = []

all_history_rows = []

for result in training_results:
    baseline = result[
        "baseline_validation_metrics"
    ]

    best = result[
        "best_validation_metrics"
    ]

    training_summary_rows.append({
        "objective":
            result["objective"],

        "objective_weight":
            result["objective_weight"],

        "best_epoch":
            result["best_epoch"],

        "baseline_validation_loss":
            baseline["loss"],

        "best_validation_loss":
            best["loss"],

        "baseline_expected_beyond_first":
            baseline[
                "expected_accepted_beyond_first"
            ],

        "best_expected_beyond_first":
            best[
                "expected_accepted_beyond_first"
            ],

        "baseline_decoder_kl":
            baseline["decoder_kl_loss"],

        "best_decoder_kl":
            best["decoder_kl_loss"],

        "baseline_local_lk":
            baseline["local_lk_loss"],

        "best_local_lk":
            best["local_lk_loss"],

        "baseline_survival_lk":
            baseline["survival_lk_loss"],

        "best_survival_lk":
            best["survival_lk_loss"],
    })

    all_history_rows.extend(
        result["history"]
    )

training_summary_df = pd.DataFrame(
    training_summary_rows
)

training_history_df = pd.DataFrame(
    all_history_rows
)

print(
    training_summary_df.to_string(
        index=False
    )
)

  objective  objective_weight  best_epoch  baseline_validation_loss  best_validation_loss  baseline_expected_beyond_first  best_expected_beyond_first  baseline_decoder_kl  best_decoder_kl  baseline_local_lk  best_local_lk  baseline_survival_lk  best_survival_lk
 decoder_kl          0.080898           0                  1.361822              1.361822                        0.749712                    0.749712             2.049413         2.049413           0.616946       0.616946              0.892898          0.892898
   local_lk          0.667756           0                  1.607997              1.607997                        0.749712                    0.749712             2.049413         2.049413           0.616946       0.616946              0.892898          0.892898
survival_lk          1.036221           2                  2.121268              2.118697                        0.749712                    0.732754             2.049413         2.041377           0.616946       0

In [44]:
training_summary_path = (
    RESULT_TABLE_DIR
    / "19_training_summary.csv"
)

training_history_path = (
    RESULT_TABLE_DIR
    / "19_training_history.csv"
)

training_summary_df.to_csv(
    training_summary_path,
    index=False,
)

training_history_df.to_csv(
    training_history_path,
    index=False,
)

print("Saved:")
print(training_summary_path)
print(training_history_path)

Saved:
/content/drive/MyDrive/ma2288_nextlat/results_v3/tables/19_training_summary.csv
/content/drive/MyDrive/ma2288_nextlat/results_v3/tables/19_training_history.csv


In [45]:
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 240)

print("=" * 100)
print("NOTEBOOK 19 — PHASE C TRAINING RESULTS")
print("=" * 100)

print(
    training_summary_df.to_string(
        index=False
    )
)

print("\nCHECKPOINTS")

print(decoder_kl_checkpoint_path)
print(local_lk_checkpoint_path)
print(survival_lk_checkpoint_path)

print("\nPHASE C COMPLETE")

NOTEBOOK 19 — PHASE C TRAINING RESULTS
  objective  objective_weight  best_epoch  baseline_validation_loss  best_validation_loss  baseline_expected_beyond_first  best_expected_beyond_first  baseline_decoder_kl  best_decoder_kl  baseline_local_lk  best_local_lk  baseline_survival_lk  best_survival_lk
 decoder_kl          0.080898           0                  1.361822              1.361822                        0.749712                    0.749712             2.049413         2.049413           0.616946       0.616946              0.892898          0.892898
   local_lk          0.667756           0                  1.607997              1.607997                        0.749712                    0.749712             2.049413         2.049413           0.616946       0.616946              0.892898          0.892898
survival_lk          1.036221           2                  2.121268              2.118697                        0.749712                    0.732754             2.049413     

In [46]:
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 260)

if "training_history_df" not in globals():
    training_history_df = pd.read_csv(
        RESULT_TABLE_DIR
        / "19_training_history.csv"
    )

history_columns = [
    "objective",
    "epoch",
    "train_loss",
    "train_latent_loss",
    "train_selected_objective_loss",
    "train_expected_beyond_first",
    "validation_loss",
    "validation_latent_loss",
    "validation_decoder_kl",
    "validation_local_lk",
    "validation_survival_lk",
    "validation_expected_beyond_first",
]

print("=" * 120)
print("FULL NOTEBOOK 19 TRAINING HISTORY")
print("=" * 120)

print(
    training_history_df[
        history_columns
    ]
    .sort_values(
        ["objective", "epoch"]
    )
    .to_string(index=False)
)

FULL NOTEBOOK 19 TRAINING HISTORY
  objective  epoch  train_loss  train_latent_loss  train_selected_objective_loss  train_expected_beyond_first  validation_loss  validation_latent_loss  validation_decoder_kl  validation_local_lk  validation_survival_lk  validation_expected_beyond_first
 decoder_kl      1    1.741355           1.578996                       2.006946                     0.743396         1.583972                1.424069               1.976596             0.620943                0.895930                          0.728490
 decoder_kl      2    1.686771           1.536611                       1.856154                     0.762324         1.522693                1.368133               1.910543             0.620123                0.895112                          0.734213
   local_lk      1    2.005444           1.592453                       0.618476                     0.743420         1.641107                1.236182               1.939673             0.606397             

In [47]:
selection_rows = []

for objective_name in VALID_OBJECTIVES:
    objective_history = (
        training_history_df[
            training_history_df["objective"]
            == objective_name
        ]
        .copy()
    )

    best_combined_row = objective_history.loc[
        objective_history[
            "validation_loss"
        ].idxmin()
    ]

    best_kl_row = objective_history.loc[
        objective_history[
            "validation_decoder_kl"
        ].idxmin()
    ]

    best_local_row = objective_history.loc[
        objective_history[
            "validation_local_lk"
        ].idxmin()
    ]

    best_survival_row = objective_history.loc[
        objective_history[
            "validation_survival_lk"
        ].idxmin()
    ]

    best_expected_row = objective_history.loc[
        objective_history[
            "validation_expected_beyond_first"
        ].idxmax()
    ]

    selection_rows.append({
        "objective": objective_name,

        "best_combined_epoch":
            int(best_combined_row["epoch"]),

        "best_decoder_kl_epoch":
            int(best_kl_row["epoch"]),

        "best_local_lk_epoch":
            int(best_local_row["epoch"]),

        "best_survival_lk_epoch":
            int(best_survival_row["epoch"]),

        "best_expected_prefix_epoch":
            int(best_expected_row["epoch"]),

        "maximum_validation_expected_beyond_first":
            best_expected_row[
                "validation_expected_beyond_first"
            ],

        "minimum_validation_decoder_kl":
            best_kl_row[
                "validation_decoder_kl"
            ],

        "minimum_validation_local_lk":
            best_local_row[
                "validation_local_lk"
            ],

        "minimum_validation_survival_lk":
            best_survival_row[
                "validation_survival_lk"
            ],
    })

selection_audit_df = pd.DataFrame(
    selection_rows
)

print(
    selection_audit_df.to_string(
        index=False
    )
)

  objective  best_combined_epoch  best_decoder_kl_epoch  best_local_lk_epoch  best_survival_lk_epoch  best_expected_prefix_epoch  maximum_validation_expected_beyond_first  minimum_validation_decoder_kl  minimum_validation_local_lk  minimum_validation_survival_lk
 decoder_kl                    2                      2                    2                       2                           2                                  0.734213                       1.910543                     0.620123                        0.895112
   local_lk                    1                      1                    1                       1                           1                                  0.746441                       1.939673                     0.606397                        0.893365
survival_lk                    2                      1                    4                       4                           4                                  0.759053                       1.957360          

In [48]:
checkpoint_audit_paths = {
    "Decoder-KL":
        decoder_kl_checkpoint_path,

    "Local-LK":
        local_lk_checkpoint_path,

    "Survival-LK":
        survival_lk_checkpoint_path,
}

checkpoint_difference_rows = []

for method_name, checkpoint_path in (
    checkpoint_audit_paths.items()
):
    checkpoint = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=False,
    )

    trained_state_dict = checkpoint[
        "model_state_dict"
    ]

    maximum_difference = max(
        (
            trained_state_dict[key].float()
            - base_state_dict[key].float()
        )
        .abs()
        .max()
        .item()
        for key in base_state_dict
    )

    parameter_l2_difference = np.sqrt(
        sum(
            (
                trained_state_dict[key].float()
                - base_state_dict[key].float()
            )
            .pow(2)
            .sum()
            .item()
            for key in base_state_dict
        )
    )

    checkpoint_difference_rows.append({
        "method": method_name,
        "best_epoch":
            checkpoint["best_epoch"],
        "maximum_parameter_difference":
            maximum_difference,
        "parameter_l2_difference":
            parameter_l2_difference,
    })

checkpoint_difference_df = pd.DataFrame(
    checkpoint_difference_rows
)

print(
    checkpoint_difference_df.to_string(
        index=False
    )
)

     method  best_epoch  maximum_parameter_difference  parameter_l2_difference
 Decoder-KL           0                      0.000000                 0.000000
   Local-LK           0                      0.000000                 0.000000
Survival-LK           2                      0.006574                 1.237998


In [49]:
C2_LEARNING_RATE = 2e-5
C2_MAX_EPOCHS = 8
C2_PATIENCE = 3

TRAIN_TRIALS_PER_PROMPT = 2
VALIDATION_TRIALS_PER_PROMPT = 3

C2_TRAIN_UNIQUE_BATCH_SIZE = 2
C2_VALIDATION_UNIQUE_BATCH_SIZE = 2

MAX_LATENT_DEGRADATION = 0.10
MIN_EXPECTED_PREFIX_IMPROVEMENT = 0.002

C2_CHECKPOINT_DIR = (
    CHECKPOINT_OUTPUT_DIR
    / "phase_c2"
)

C2_CHECKPOINT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print("Learning rate:", C2_LEARNING_RATE)
print("Train trials:", TRAIN_TRIALS_PER_PROMPT)
print(
    "Validation trials:",
    VALIDATION_TRIALS_PER_PROMPT,
)
print(
    "Maximum latent degradation:",
    MAX_LATENT_DEGRADATION,
)
print("Checkpoint directory:")
print(C2_CHECKPOINT_DIR)

Learning rate: 2e-05
Train trials: 2
Validation trials: 3
Maximum latent degradation: 0.1
Checkpoint directory:
/content/drive/MyDrive/ma2288_nextlat/checkpoints_v3/phase_c2


In [50]:
def create_c2_train_loader(seed):
    generator = torch.Generator()
    generator.manual_seed(seed)

    return DataLoader(
        train_dataset,
        batch_size=C2_TRAIN_UNIQUE_BATCH_SIZE,
        shuffle=True,
        num_workers=0,
        generator=generator,
    )


c2_validation_loader = DataLoader(
    validation_dataset,
    batch_size=C2_VALIDATION_UNIQUE_BATCH_SIZE,
    shuffle=False,
    num_workers=0,
)

In [51]:
@torch.no_grad()
def evaluate_c2(
    transition_model,
    objective_name,
    objective_weight,
):
    transition_model.eval()

    total_chains = 0

    scalar_keys = [
        "loss",
        "latent_loss",
        "selected_objective_loss",
        "decoder_kl_loss",
        "local_lk_loss",
        "survival_lk_loss",
        "expected_accepted_beyond_first",
    ]

    sums = {
        key: 0.0
        for key in scalar_keys
    }

    acceptance_sum = None
    survival_sum = None

    for batch_index, prompt_batch in enumerate(
        c2_validation_loader
    ):
        repeated_prompts = (
            prompt_batch.repeat_interleave(
                VALIDATION_TRIALS_PER_PROMPT,
                dim=0,
            )
        )

        metrics = compute_onpolicy_training_loss(
            transition_model=transition_model,
            prompt_batch=repeated_prompts,
            objective_name=objective_name,
            rollout_length=ROLLOUT_LENGTH,
            latent_weight=1.0,
            objective_weight=objective_weight,
            sampling_seed=(
                900000 + batch_index
            ),
        )

        num_chains = repeated_prompts.shape[0]
        total_chains += num_chains

        for key in scalar_keys:
            sums[key] += (
                metrics[key].item()
                * num_chains
            )

        current_acceptance = (
            metrics["acceptance_by_step"]
            .detach()
            .cpu()
            * num_chains
        )

        current_survival = (
            metrics[
                "survival_by_trainable_step"
            ]
            .detach()
            .cpu()
            * num_chains
        )

        if acceptance_sum is None:
            acceptance_sum = current_acceptance
            survival_sum = current_survival
        else:
            acceptance_sum += current_acceptance
            survival_sum += current_survival

    result = {
        key: value / total_chains
        for key, value in sums.items()
    }

    result["acceptance_by_step"] = (
        acceptance_sum / total_chains
    ).numpy()

    result["survival_by_step"] = (
        survival_sum / total_chains
    ).numpy()

    return result

In [52]:
def train_c2_objective(
    objective_name,
    objective_weight,
):
    print("\n" + "=" * 100)
    print("PHASE C2:", objective_name)
    print("=" * 100)

    model = ResidualLatentTransition(
        hidden_size=HIDDEN_SIZE,
        token_embedding_size=HIDDEN_SIZE,
        intermediate_size=512,
    ).to(device)

    model.load_state_dict(
        base_state_dict,
        strict=True,
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=C2_LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )

    baseline_metrics = evaluate_c2(
        transition_model=model,
        objective_name=objective_name,
        objective_weight=objective_weight,
    )

    baseline_latent = baseline_metrics[
        "latent_loss"
    ]

    maximum_allowed_latent = (
        baseline_latent
        * (1.0 + MAX_LATENT_DEGRADATION)
    )

    best_score = baseline_metrics[
        "expected_accepted_beyond_first"
    ]

    best_epoch = 0
    best_state = clone_state_dict_to_cpu(model)
    best_metrics = copy.deepcopy(
        baseline_metrics
    )

    epochs_without_improvement = 0
    history = []

    print(
        "Baseline | "
        f"latent={baseline_latent:.6f} | "
        f"E[beyond1]={best_score:.6f} | "
        f"max_allowed_latent="
        f"{maximum_allowed_latent:.6f}"
    )

    for epoch in range(
        1,
        C2_MAX_EPOCHS + 1,
    ):
        model.train()

        train_loader_c2 = create_c2_train_loader(
            SEED + epoch
        )

        train_chains = 0
        train_loss_sum = 0.0

        for batch_index, prompt_batch in enumerate(
            train_loader_c2
        ):
            repeated_prompts = (
                prompt_batch.repeat_interleave(
                    TRAIN_TRIALS_PER_PROMPT,
                    dim=0,
                )
            )

            optimizer.zero_grad(
                set_to_none=True
            )

            metrics = compute_onpolicy_training_loss(
                transition_model=model,
                prompt_batch=repeated_prompts,
                objective_name=objective_name,
                rollout_length=ROLLOUT_LENGTH,
                latent_weight=1.0,
                objective_weight=objective_weight,
                sampling_seed=(
                    SEED
                    + epoch * 10000
                    + batch_index
                ),
            )

            loss = metrics["loss"]

            assert torch.isfinite(loss)

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                GRADIENT_CLIP,
            )

            optimizer.step()

            num_chains = repeated_prompts.shape[0]
            train_chains += num_chains
            train_loss_sum += (
                loss.item() * num_chains
            )

        validation_metrics = evaluate_c2(
            transition_model=model,
            objective_name=objective_name,
            objective_weight=objective_weight,
        )

        validation_expected = (
            validation_metrics[
                "expected_accepted_beyond_first"
            ]
        )

        validation_latent = (
            validation_metrics["latent_loss"]
        )

        latent_feasible = (
            validation_latent
            <= maximum_allowed_latent
        )

        history.append({
            "objective": objective_name,
            "epoch": epoch,
            "train_loss":
                train_loss_sum / train_chains,
            "validation_loss":
                validation_metrics["loss"],
            "validation_latent_loss":
                validation_latent,
            "validation_decoder_kl":
                validation_metrics[
                    "decoder_kl_loss"
                ],
            "validation_local_lk":
                validation_metrics[
                    "local_lk_loss"
                ],
            "validation_survival_lk":
                validation_metrics[
                    "survival_lk_loss"
                ],
            "validation_expected_beyond_first":
                validation_expected,
            "latent_feasible":
                latent_feasible,
        })

        epoch_path = (
            C2_CHECKPOINT_DIR
            / (
                f"{objective_name}"
                f"_epoch{epoch}_seed42.pt"
            )
        )

        torch.save(
            {
                "model_state_dict":
                    clone_state_dict_to_cpu(model),
                "objective":
                    objective_name,
                "objective_weight":
                    objective_weight,
                "epoch":
                    epoch,
                "validation_metrics":
                    validation_metrics,
                "latent_feasible":
                    latent_feasible,
                "seed":
                    SEED,
            },
            epoch_path,
        )

        print(
            f"Epoch {epoch:02d} | "
            f"train={train_loss_sum / train_chains:.6f} | "
            f"latent={validation_latent:.6f} | "
            f"E[beyond1]={validation_expected:.6f} | "
            f"feasible={latent_feasible}"
        )

        improved = (
            latent_feasible
            and validation_expected
            > best_score
            + MIN_EXPECTED_PREFIX_IMPROVEMENT
        )

        if improved:
            best_score = validation_expected
            best_epoch = epoch
            best_state = clone_state_dict_to_cpu(
                model
            )
            best_metrics = copy.deepcopy(
                validation_metrics
            )
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if (
            epochs_without_improvement
            >= C2_PATIENCE
        ):
            print(
                "Early stopping at epoch",
                epoch,
            )
            break

    model.load_state_dict(
        best_state,
        strict=True,
    )

    selected_path = (
        CHECKPOINT_OUTPUT_DIR
        / (
            f"{objective_name}"
            f"_c2_selected_seed42.pt"
        )
    )

    torch.save(
        {
            "model_state_dict": best_state,
            "objective": objective_name,
            "objective_weight":
                objective_weight,
            "best_epoch": best_epoch,
            "baseline_metrics":
                baseline_metrics,
            "best_metrics":
                best_metrics,
            "history": history,
            "selection_metric":
                "expected_accepted_beyond_first",
            "maximum_latent_degradation":
                MAX_LATENT_DEGRADATION,
            "seed": SEED,
        },
        selected_path,
    )

    print("\nSelected epoch:", best_epoch)
    print("Selected expected beyond first:",
          best_score)
    print("Saved:", selected_path)

    return {
        "model": model,
        "objective": objective_name,
        "best_epoch": best_epoch,
        "baseline_metrics":
            baseline_metrics,
        "best_metrics": best_metrics,
        "history": history,
        "checkpoint_path":
            selected_path,
    }

In [53]:
decoder_kl_c2 = train_c2_objective(
    "decoder_kl",
    calibrated_objective_weights[
        "decoder_kl"
    ],
)


PHASE C2: decoder_kl
Baseline | latent=1.400251 | E[beyond1]=0.737883 | max_allowed_latent=1.540276
Epoch 01 | train=1.645655 | latent=1.370644 | E[beyond1]=0.756804 | feasible=True
Epoch 02 | train=1.688070 | latent=1.426567 | E[beyond1]=0.757636 | feasible=True
Epoch 03 | train=1.555744 | latent=1.382700 | E[beyond1]=0.760951 | feasible=True
Epoch 04 | train=1.547717 | latent=1.409710 | E[beyond1]=0.763779 | feasible=True
Epoch 05 | train=1.485035 | latent=1.443871 | E[beyond1]=0.761308 | feasible=True
Epoch 06 | train=1.514331 | latent=1.412579 | E[beyond1]=0.763979 | feasible=True
Epoch 07 | train=1.514541 | latent=1.407686 | E[beyond1]=0.757493 | feasible=True
Early stopping at epoch 7

Selected epoch: 4
Selected expected beyond first: 0.7637785330414772
Saved: /content/drive/MyDrive/ma2288_nextlat/checkpoints_v3/decoder_kl_c2_selected_seed42.pt


In [54]:
local_lk_c2 = train_c2_objective(
    "local_lk",
    calibrated_objective_weights[
        "local_lk"
    ],
)


PHASE C2: local_lk
Baseline | latent=1.400251 | E[beyond1]=0.737883 | max_allowed_latent=1.540276
Epoch 01 | train=1.871696 | latent=1.367521 | E[beyond1]=0.758854 | feasible=True
Epoch 02 | train=1.924101 | latent=1.319764 | E[beyond1]=0.766585 | feasible=True
Epoch 03 | train=1.875834 | latent=1.406074 | E[beyond1]=0.760838 | feasible=True
Epoch 04 | train=1.754136 | latent=1.366638 | E[beyond1]=0.762648 | feasible=True
Epoch 05 | train=1.722449 | latent=1.405234 | E[beyond1]=0.766151 | feasible=True
Early stopping at epoch 5

Selected epoch: 2
Selected expected beyond first: 0.7665851049125194
Saved: /content/drive/MyDrive/ma2288_nextlat/checkpoints_v3/local_lk_c2_selected_seed42.pt


In [55]:
survival_lk_c2 = train_c2_objective(
    "survival_lk",
    calibrated_objective_weights[
        "survival_lk"
    ],
)


PHASE C2: survival_lk
Baseline | latent=1.400251 | E[beyond1]=0.737883 | max_allowed_latent=1.540276
Epoch 01 | train=2.363052 | latent=1.367674 | E[beyond1]=0.753329 | feasible=True
Epoch 02 | train=2.464732 | latent=1.407993 | E[beyond1]=0.760160 | feasible=True
Epoch 03 | train=2.361627 | latent=1.445129 | E[beyond1]=0.757402 | feasible=True
Epoch 04 | train=2.283642 | latent=1.451538 | E[beyond1]=0.762224 | feasible=True
Epoch 05 | train=2.242795 | latent=1.387730 | E[beyond1]=0.764750 | feasible=True
Epoch 06 | train=2.195796 | latent=1.480462 | E[beyond1]=0.762382 | feasible=True
Epoch 07 | train=2.245337 | latent=1.428910 | E[beyond1]=0.768686 | feasible=True
Epoch 08 | train=2.432199 | latent=1.317305 | E[beyond1]=0.767916 | feasible=True

Selected epoch: 7
Selected expected beyond first: 0.7686856545507907
Saved: /content/drive/MyDrive/ma2288_nextlat/checkpoints_v3/survival_lk_c2_selected_seed42.pt


In [56]:
c2_results = [
    decoder_kl_c2,
    local_lk_c2,
    survival_lk_c2,
]

c2_summary_rows = []

for result in c2_results:
    baseline = result["baseline_metrics"]
    best = result["best_metrics"]

    c2_summary_rows.append({
        "objective":
            result["objective"],
        "best_epoch":
            result["best_epoch"],
        "baseline_latent":
            baseline["latent_loss"],
        "best_latent":
            best["latent_loss"],
        "baseline_expected_beyond_first":
            baseline[
                "expected_accepted_beyond_first"
            ],
        "best_expected_beyond_first":
            best[
                "expected_accepted_beyond_first"
            ],
        "baseline_local_lk":
            baseline["local_lk_loss"],
        "best_local_lk":
            best["local_lk_loss"],
        "baseline_survival_lk":
            baseline["survival_lk_loss"],
        "best_survival_lk":
            best["survival_lk_loss"],
        "checkpoint":
            str(result["checkpoint_path"]),
    })

c2_summary_df = pd.DataFrame(
    c2_summary_rows
)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 260)

print("=" * 110)
print("NOTEBOOK 19 — PHASE C2 RESULTS")
print("=" * 110)

print(
    c2_summary_df.to_string(
        index=False
    )
)

NOTEBOOK 19 — PHASE C2 RESULTS
  objective  best_epoch  baseline_latent  best_latent  baseline_expected_beyond_first  best_expected_beyond_first  baseline_local_lk  best_local_lk  baseline_survival_lk  best_survival_lk                                                                             checkpoint
 decoder_kl           4         1.400251     1.409710                        0.737883                    0.763779           0.624206       0.609606              0.894588          0.890889  /content/drive/MyDrive/ma2288_nextlat/checkpoints_v3/decoder_kl_c2_selected_seed42.pt
   local_lk           2         1.400251     1.319764                        0.737883                    0.766585           0.624206       0.608516              0.894588          0.890488    /content/drive/MyDrive/ma2288_nextlat/checkpoints_v3/local_lk_c2_selected_seed42.pt
survival_lk           7         1.400251     1.428910                        0.737883                    0.768686           0.624206       0.604

In [57]:
PHASE_D_CHECKPOINTS = {
    "Multi-step": (
        PROJECT_DIR
        / "checkpoints"
        / "multistep_transition_seed42.pt"
    ),

    "Decoder-KL": (
        PROJECT_DIR
        / "checkpoints_v3"
        / "decoder_kl_c2_selected_seed42.pt"
    ),

    "Local-LK": (
        PROJECT_DIR
        / "checkpoints_v3"
        / "local_lk_c2_selected_seed42.pt"
    ),

    "Survival-LK": (
        PROJECT_DIR
        / "checkpoints_v3"
        / "survival_lk_c2_selected_seed42.pt"
    ),
}

phase_d_models = {}

for method_name, checkpoint_path in (
    PHASE_D_CHECKPOINTS.items()
):
    assert checkpoint_path.exists(), (
        f"Missing: {checkpoint_path}"
    )

    checkpoint = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=False,
    )

    model = ResidualLatentTransition(
        hidden_size=HIDDEN_SIZE,
        token_embedding_size=HIDDEN_SIZE,
        intermediate_size=512,
    ).to(device)

    model.load_state_dict(
        checkpoint["model_state_dict"],
        strict=True,
    )

    model.eval()

    for parameter in model.parameters():
        parameter.requires_grad = False

    phase_d_models[method_name] = model

    print(
        method_name,
        "loaded from",
        checkpoint_path.name,
    )

Multi-step loaded from multistep_transition_seed42.pt
Decoder-KL loaded from decoder_kl_c2_selected_seed42.pt
Local-LK loaded from local_lk_c2_selected_seed42.pt
Survival-LK loaded from survival_lk_c2_selected_seed42.pt


In [58]:
PHASE_D_TRIALS = 5
PHASE_D_ROLLOUT_LENGTH = 8
PHASE_D_BATCH_SIZE = 10

phase_d_prompt_tokens = (
    test_prompt_tokens
    .repeat_interleave(
        PHASE_D_TRIALS,
        dim=0,
    )
)

phase_d_prompt_indices = (
    torch.arange(
        test_prompt_tokens.shape[0]
    )
    .repeat_interleave(
        PHASE_D_TRIALS
    )
)

phase_d_trial_indices = (
    torch.arange(
        PHASE_D_TRIALS
    )
    .repeat(
        test_prompt_tokens.shape[0]
    )
)

print(
    "Prompt chains:",
    phase_d_prompt_tokens.shape,
)

print(
    "Prompt indices:",
    phase_d_prompt_indices.shape,
)

print(
    "Trial indices:",
    phase_d_trial_indices.shape,
)

Prompt chains: torch.Size([400, 32])
Prompt indices: torch.Size([400])
Trial indices: torch.Size([400])


In [59]:
@torch.inference_mode()
def evaluate_phase_d_model(
    method_name,
    transition_model,
    prompt_tokens,
    prompt_indices,
    trial_indices,
    batch_size=10,
    rollout_length=8,
    seed=42,
    epsilon=1e-8,
):
    transition_model.eval()

    num_chains = prompt_tokens.shape[0]
    records = []

    sampling_generator = torch.Generator(
        device=device.type
    )

    sampling_generator.manual_seed(
        seed + 1000
    )

    acceptance_generator = torch.Generator(
        device=device.type
    )

    acceptance_generator.manual_seed(
        seed + 2000
    )

    for batch_start in range(
        0,
        num_chains,
        batch_size,
    ):
        batch_end = min(
            batch_start + batch_size,
            num_chains,
        )

        current_context = prompt_tokens[
            batch_start:batch_end
        ].to(device)

        current_prompt_indices = (
            prompt_indices[
                batch_start:batch_end
            ]
        )

        current_trial_indices = (
            trial_indices[
                batch_start:batch_end
            ]
        )

        current_batch_size = (
            current_context.shape[0]
        )

        target_hidden, target_logits = (
            get_target_state_and_logits(
                current_context
            )
        )

        draft_hidden = target_hidden.clone()

        empirical_alive = torch.ones(
            current_batch_size,
            dtype=torch.bool,
            device=device,
        )

        expected_survival = torch.ones(
            current_batch_size,
            dtype=torch.float32,
            device=device,
        )

        for step in range(
            rollout_length
        ):
            draft_logits = target_model.lm_head(
                draft_hidden
            ).float()

            target_log_probability = (
                F.log_softmax(
                    target_logits,
                    dim=-1,
                )
            )

            draft_log_probability = (
                F.log_softmax(
                    draft_logits,
                    dim=-1,
                )
            )

            target_probability = (
                target_log_probability.exp()
            )

            draft_probability = (
                draft_log_probability.exp()
            )

            sampled_token_ids = (
                torch.multinomial(
                    draft_probability,
                    num_samples=1,
                    replacement=True,
                    generator=sampling_generator,
                )
                .squeeze(1)
            )

            selected_target_probability = (
                target_probability
                .gather(
                    1,
                    sampled_token_ids.unsqueeze(1),
                )
                .squeeze(1)
            )

            selected_draft_probability = (
                draft_probability
                .gather(
                    1,
                    sampled_token_ids.unsqueeze(1),
                )
                .squeeze(1)
            )

            token_acceptance_probability = (
                torch.minimum(
                    torch.ones_like(
                        selected_target_probability
                    ),
                    selected_target_probability
                    / selected_draft_probability
                    .clamp_min(epsilon),
                )
            )

            expected_survival = (
                expected_survival
                * token_acceptance_probability
            )

            acceptance_uniform = torch.rand(
                current_batch_size,
                device=device,
                generator=acceptance_generator,
            )

            accepted_at_step = (
                acceptance_uniform
                < token_acceptance_probability
            )

            empirical_alive = (
                empirical_alive
                & accepted_at_step
            )

            output_kl = torch.sum(
                target_probability
                * (
                    target_log_probability
                    - draft_log_probability
                ),
                dim=-1,
            )

            top1_agreement = (
                target_logits.argmax(dim=-1)
                == draft_logits.argmax(dim=-1)
            )

            for local_index in range(
                current_batch_size
            ):
                records.append({
                    "method": method_name,

                    "chain_index":
                        batch_start
                        + local_index,

                    "prompt_index":
                        int(
                            current_prompt_indices[
                                local_index
                            ]
                        ),

                    "trial_index":
                        int(
                            current_trial_indices[
                                local_index
                            ]
                        ),

                    "step":
                        step + 1,

                    "draft_token_id":
                        int(
                            sampled_token_ids[
                                local_index
                            ].cpu()
                        ),

                    "token_acceptance_probability":
                        float(
                            token_acceptance_probability[
                                local_index
                            ].cpu()
                        ),

                    "expected_survival":
                        float(
                            expected_survival[
                                local_index
                            ].cpu()
                        ),

                    "accepted_at_step":
                        bool(
                            accepted_at_step[
                                local_index
                            ].cpu()
                        ),

                    "empirical_survival":
                        bool(
                            empirical_alive[
                                local_index
                            ].cpu()
                        ),

                    "output_kl":
                        float(
                            output_kl[
                                local_index
                            ].cpu()
                        ),

                    "top1_agreement":
                        bool(
                            top1_agreement[
                                local_index
                            ].cpu()
                        ),
                })

            if step == rollout_length - 1:
                continue

            token_embeddings = token_embedding(
                sampled_token_ids
            ).float()

            draft_hidden = transition_model(
                draft_hidden,
                token_embeddings,
            )

            current_context = torch.cat(
                [
                    current_context,
                    sampled_token_ids.unsqueeze(1),
                ],
                dim=1,
            )

            target_hidden, target_logits = (
                get_target_state_and_logits(
                    current_context
                )
            )

        if (
            batch_end % 100 == 0
            or batch_end == num_chains
        ):
            print(
                f"{method_name}: "
                f"{batch_end}/{num_chains}"
            )

    return pd.DataFrame(records)

In [60]:
small_phase_d_records = (
    evaluate_phase_d_model(
        method_name="Survival-LK test",
        transition_model=phase_d_models[
            "Survival-LK"
        ],
        prompt_tokens=(
            test_prompt_tokens[:2]
            .repeat_interleave(2, dim=0)
        ),
        prompt_indices=(
            torch.arange(2)
            .repeat_interleave(2)
        ),
        trial_indices=(
            torch.arange(2).repeat(2)
        ),
        batch_size=4,
        rollout_length=8,
        seed=SEED,
    )
)

step_1_test = small_phase_d_records[
    small_phase_d_records["step"] == 1
]

print(
    step_1_test[
        [
            "output_kl",
            "token_acceptance_probability",
            "expected_survival",
            "empirical_survival",
        ]
    ].to_string(index=False)
)

assert step_1_test[
    "output_kl"
].max() < 1e-4

assert step_1_test[
    "token_acceptance_probability"
].min() > 0.999

print("Phase D sanity test passed.")

Survival-LK test: 4/4
 output_kl  token_acceptance_probability  expected_survival  empirical_survival
       0.0                           1.0                1.0                True
       0.0                           1.0                1.0                True
       0.0                           1.0                1.0                True
       0.0                           1.0                1.0                True
Phase D sanity test passed.


In [61]:
phase_d_record_tables = []

for method_name, model in (
    phase_d_models.items()
):
    print("\n" + "=" * 90)
    print("Evaluating:", method_name)

    method_records = (
        evaluate_phase_d_model(
            method_name=method_name,
            transition_model=model,
            prompt_tokens=
                phase_d_prompt_tokens,
            prompt_indices=
                phase_d_prompt_indices,
            trial_indices=
                phase_d_trial_indices,
            batch_size=
                PHASE_D_BATCH_SIZE,
            rollout_length=
                PHASE_D_ROLLOUT_LENGTH,
            seed=SEED,
        )
    )

    phase_d_record_tables.append(
        method_records
    )

phase_d_records_df = pd.concat(
    phase_d_record_tables,
    ignore_index=True,
)

print(
    "Record shape:",
    phase_d_records_df.shape,
)

assert len(phase_d_records_df) == (
    4 * 400 * 8
)


Evaluating: Multi-step
Multi-step: 100/400
Multi-step: 200/400
Multi-step: 300/400
Multi-step: 400/400

Evaluating: Decoder-KL
Decoder-KL: 100/400
Decoder-KL: 200/400
Decoder-KL: 300/400
Decoder-KL: 400/400

Evaluating: Local-LK
Local-LK: 100/400
Local-LK: 200/400
Local-LK: 300/400
Local-LK: 400/400

Evaluating: Survival-LK
Survival-LK: 100/400
Survival-LK: 200/400
Survival-LK: 300/400
Survival-LK: 400/400
Record shape: (12800, 12)


In [62]:
phase_d_step_summary = (
    phase_d_records_df
    .groupby(["method", "step"])
    .agg(
        mean_output_kl=(
            "output_kl",
            "mean",
        ),

        mean_token_acceptance=(
            "token_acceptance_probability",
            "mean",
        ),

        mean_expected_survival=(
            "expected_survival",
            "mean",
        ),

        empirical_survival_rate=(
            "empirical_survival",
            "mean",
        ),

        top1_agreement=(
            "top1_agreement",
            "mean",
        ),
    )
    .reset_index()
)

print(
    phase_d_step_summary
    .sort_values(["step", "method"])
    .to_string(index=False)
)

     method  step  mean_output_kl  mean_token_acceptance  mean_expected_survival  empirical_survival_rate  top1_agreement
 Decoder-KL     1        0.000000               1.000000                1.000000                   1.0000          1.0000
   Local-LK     1        0.000000               1.000000                1.000000                   1.0000          1.0000
 Multi-step     1        0.000000               1.000000                1.000000                   1.0000          1.0000
Survival-LK     1        0.000000               1.000000                1.000000                   1.0000          1.0000
 Decoder-KL     2        1.451467               0.497815                0.497815                   0.5275          0.3425
   Local-LK     2        1.476151               0.499647                0.499647                   0.5225          0.3400
 Multi-step     2        1.515425               0.485318                0.485318                   0.5050          0.3200
Survival-LK     2       

In [63]:
GAMMAS = [2, 4, 8]

chain_gamma_rows = []

grouped_chains = (
    phase_d_records_df
    .groupby(
        [
            "method",
            "chain_index",
            "prompt_index",
            "trial_index",
        ]
    )
)

for group_keys, chain_df in grouped_chains:
    (
        method_name,
        chain_index,
        prompt_index,
        trial_index,
    ) = group_keys

    chain_df = chain_df.sort_values(
        "step"
    )

    for gamma in GAMMAS:
        selected_steps = chain_df[
            chain_df["step"] <= gamma
        ]

        expected_prefix = (
            selected_steps[
                "expected_survival"
            ].sum()
        )

        empirical_prefix = (
            selected_steps[
                "empirical_survival"
            ].sum()
        )

        full_empirical_acceptance = bool(
            selected_steps[
                "empirical_survival"
            ].iloc[-1]
        )

        chain_gamma_rows.append({
            "method": method_name,
            "chain_index": chain_index,
            "prompt_index": prompt_index,
            "trial_index": trial_index,
            "gamma": gamma,
            "expected_accepted_prefix":
                expected_prefix,
            "empirical_accepted_prefix":
                empirical_prefix,
            "full_acceptance":
                full_empirical_acceptance,
        })

phase_d_chain_gamma_df = pd.DataFrame(
    chain_gamma_rows
)

phase_d_gamma_summary = (
    phase_d_chain_gamma_df
    .groupby(["method", "gamma"])
    .agg(
        num_chains=(
            "chain_index",
            "size",
        ),

        mean_expected_prefix=(
            "expected_accepted_prefix",
            "mean",
        ),

        mean_empirical_prefix=(
            "empirical_accepted_prefix",
            "mean",
        ),

        std_empirical_prefix=(
            "empirical_accepted_prefix",
            "std",
        ),

        full_acceptance_rate=(
            "full_acceptance",
            "mean",
        ),
    )
    .reset_index()
)

print(
    phase_d_gamma_summary
    .sort_values(["gamma", "method"])
    .to_string(index=False)
)

     method  gamma  num_chains  mean_expected_prefix  mean_empirical_prefix  std_empirical_prefix  full_acceptance_rate
 Decoder-KL      2         400              1.497815                 1.5275              0.499868                0.5275
   Local-LK      2         400              1.499647                 1.5225              0.500119                0.5225
 Multi-step      2         400              1.485318                 1.5050              0.500601                0.5050
Survival-LK      2         400              1.493837                 1.5150              0.500401                0.5150
 Decoder-KL      4         400              1.773349                 1.8300              0.963546                0.0900
   Local-LK      4         400              1.772522                 1.8125              0.951107                0.0850
 Multi-step      4         400              1.753499                 1.8050              0.969187                0.0900
Survival-LK      4         400          

In [64]:
def phase_d_paired_bootstrap(
    chain_gamma_df,
    method_a,
    method_b,
    gamma,
    metric,
    num_bootstrap=10000,
    seed=42,
):
    selected = chain_gamma_df[
        chain_gamma_df["gamma"] == gamma
    ]

    prompt_level = (
        selected
        .groupby(
            ["method", "prompt_index"]
        )[metric]
        .mean()
        .unstack("method")
    )

    paired = prompt_level[
        [method_a, method_b]
    ].dropna()

    differences = (
        paired[method_a]
        - paired[method_b]
    ).to_numpy()

    rng = np.random.default_rng(seed)

    bootstrap_means = np.empty(
        num_bootstrap
    )

    for index in range(num_bootstrap):
        sampled_indices = rng.integers(
            0,
            len(differences),
            size=len(differences),
        )

        bootstrap_means[index] = (
            differences[
                sampled_indices
            ].mean()
        )

    return {
        "method_a": method_a,
        "method_b": method_b,
        "gamma": gamma,
        "metric": metric,
        "point_estimate":
            differences.mean(),
        "ci_lower":
            np.percentile(
                bootstrap_means,
                2.5,
            ),
        "ci_upper":
            np.percentile(
                bootstrap_means,
                97.5,
            ),
        "probability_positive":
            np.mean(
                bootstrap_means > 0
            ),
    }

In [65]:
phase_d_bootstrap_rows = []

for gamma in GAMMAS:
    for metric in [
        "expected_accepted_prefix",
        "empirical_accepted_prefix",
    ]:
        for baseline_method in [
            "Multi-step",
            "Decoder-KL",
            "Local-LK",
        ]:
            phase_d_bootstrap_rows.append(
                phase_d_paired_bootstrap(
                    chain_gamma_df=
                        phase_d_chain_gamma_df,
                    method_a=
                        "Survival-LK",
                    method_b=
                        baseline_method,
                    gamma=gamma,
                    metric=metric,
                    num_bootstrap=10000,
                    seed=SEED + gamma,
                )
            )

phase_d_bootstrap_df = pd.DataFrame(
    phase_d_bootstrap_rows
)

print(
    phase_d_bootstrap_df
    .sort_values(
        ["metric", "gamma", "method_b"]
    )
    .to_string(index=False)
)

   method_a   method_b  gamma                    metric  point_estimate  ci_lower  ci_upper  probability_positive
Survival-LK Decoder-KL      2 empirical_accepted_prefix       -0.012500 -0.037500  0.012500                0.1316
Survival-LK   Local-LK      2 empirical_accepted_prefix       -0.007500 -0.030000  0.012500                0.2106
Survival-LK Multi-step      2 empirical_accepted_prefix        0.010000 -0.020000  0.037562                0.7218
Survival-LK Decoder-KL      4 empirical_accepted_prefix       -0.015000 -0.062500  0.032500                0.2537
Survival-LK   Local-LK      4 empirical_accepted_prefix        0.002500 -0.040000  0.045000                0.5259
Survival-LK Multi-step      4 empirical_accepted_prefix        0.010000 -0.050000  0.070062                0.6146
Survival-LK Decoder-KL      8 empirical_accepted_prefix       -0.027500 -0.087500  0.027500                0.1646
Survival-LK   Local-LK      8 empirical_accepted_prefix       -0.012500 -0.067500  0.040

In [66]:
phase_d_records_df.to_csv(
    RESULT_TABLE_DIR
    / "19_phase_d_step_records.csv",
    index=False,
)

phase_d_step_summary.to_csv(
    RESULT_TABLE_DIR
    / "19_phase_d_step_summary.csv",
    index=False,
)

phase_d_chain_gamma_df.to_csv(
    RESULT_TABLE_DIR
    / "19_phase_d_chain_gamma.csv",
    index=False,
)

phase_d_gamma_summary.to_csv(
    RESULT_TABLE_DIR
    / "19_phase_d_gamma_summary.csv",
    index=False,
)

phase_d_bootstrap_df.to_csv(
    RESULT_TABLE_DIR
    / "19_phase_d_bootstrap.csv",
    index=False,
)

print("Phase D results saved.")

Phase D results saved.


In [67]:
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 260)

print("=" * 110)
print("TABLE 1: PHASE D GAMMA SUMMARY")
print("=" * 110)

print(
    phase_d_gamma_summary
    .sort_values(["gamma", "method"])
    .to_string(index=False)
)

print("\n" + "=" * 110)
print("TABLE 2: PHASE D STEP SUMMARY")
print("=" * 110)

print(
    phase_d_step_summary
    .sort_values(["step", "method"])
    .to_string(index=False)
)

print("\n" + "=" * 110)
print("TABLE 3: PHASE D BOOTSTRAP")
print("=" * 110)

print(
    phase_d_bootstrap_df
    .sort_values(
        ["metric", "gamma", "method_b"]
    )
    .to_string(index=False)
)

print("\nPHASE D COMPLETE")

TABLE 1: PHASE D GAMMA SUMMARY
     method  gamma  num_chains  mean_expected_prefix  mean_empirical_prefix  std_empirical_prefix  full_acceptance_rate
 Decoder-KL      2         400              1.497815                 1.5275              0.499868                0.5275
   Local-LK      2         400              1.499647                 1.5225              0.500119                0.5225
 Multi-step      2         400              1.485318                 1.5050              0.500601                0.5050
Survival-LK      2         400              1.493837                 1.5150              0.500401                0.5150
 Decoder-KL      4         400              1.773349                 1.8300              0.963546                0.0900
   Local-LK      4         400              1.772522                 1.8125              0.951107                0.0850
 Multi-step      4         400              1.753499                 1.8050              0.969187                0.0900
Survival-